# 01b — Validasi Relevansi & Pembentukan Corpus Modeling
### Dashboard Identifikasi dan Analisis Tren Isu Kesehatan Masyarakat di Bangkalan (SBERT-BERTopic)

**Notebook ini BUKAN untuk menjalankan SBERT atau BERTopic.** Tujuannya:

1. Memvalidasi relevansi berita yang heuristiknya masih ambigu (`perlu_pemeriksaan_manual`).
2. Menyediakan sampel quality-control untuk memeriksa heuristik `kemungkinan_relevan` /
   `kemungkinan_tidak_relevan` (berita) dan `is_health_related` (ulasan).
3. Menganalisis panjang review dan menandai (bukan menghapus) teks yang terlalu pendek untuk topic modeling.
4. Menambahkan kolom waktu (`tahun`, `bulan`, `tahun_bulan`) dan `document_id` yang reproducible.
5. Setelah validasi manual diisi, membentuk dua corpus terpisah yang siap dipakai
   `02_sbert_bertopic.ipynb`: `berita_modeling.csv` dan `ulasan_modeling.csv`.
6. Membuat laporan kualitas akhir (`final_validation_report.csv`).

**Prinsip yang dipegang sepanjang notebook ini:**
- Keyword/heuristik BUKAN ground truth — setiap keputusan filtering harus bisa dijelaskan.
- Tidak ada keputusan relevansi otomatis yang final tanpa jejak (`sumber_keputusan` dicatat).
- Dataset `raw` dan `cleaned` tidak pernah ditimpa. `data/modeling/` selalu dataset baru.
- Tidak ada stemming / stopword removal / perubahan makna teks di notebook ini.
- Semua sampling memakai `random_state=42` agar reproducible.

> **Catatan urutan:** instruksi asli mendaftar "Bagian 5" (kolom waktu) dan "Bagian 6"
> (`document_id`) setelah Bagian 1-4. Di notebook ini keduanya sengaja dikerjakan **lebih awal**
> (langsung setelah data dimuat), karena `document_id` dipakai sebagai `annotation_id` yang
> reproducible pada file-file validasi di Bagian 1-3. Semua Bagian tetap ada, hanya urutan
> eksekusi disusun ulang agar konsisten.


## 0. Import Library & Konfigurasi Path

In [ ]:
import pandas as pd
import numpy as np
import os
import json as json_lib
from datetime import datetime

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42  # dipakai di SEMUA sampling pada notebook ini, agar reproducible
print("Library siap. RANDOM_STATE =", RANDOM_STATE)


In [ ]:
# Jalankan sel ini HANYA di Google Colab. Lewati jika menjalankan secara lokal.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
BASE_DIR = "/content/drive/MyDrive/project"

CLEANED_DIR = os.path.join(BASE_DIR, "data", "cleaned")
VALIDATION_DIR = os.path.join(BASE_DIR, "data", "validation")
MODELING_DIR = os.path.join(BASE_DIR, "data", "modeling")
os.makedirs(VALIDATION_DIR, exist_ok=True)
os.makedirs(MODELING_DIR, exist_ok=True)

NEWS_CLEANED_PATH = os.path.join(CLEANED_DIR, "berita_bangkalan_cleaned.csv")
REVIEW_CLEANED_PATH = os.path.join(CLEANED_DIR, "ulasan_bangkalan_cleaned.csv")
CLEANING_LOG_PATH = os.path.join(CLEANED_DIR, "cleaning_log.json")

BERITA_MANUAL_XLSX = os.path.join(VALIDATION_DIR, "berita_manual_review.xlsx")
BERITA_MANUAL_CSV = os.path.join(VALIDATION_DIR, "berita_manual_review.csv")
BERITA_SAMPLE_CHECK_XLSX = os.path.join(VALIDATION_DIR, "berita_sample_check.xlsx")
REVIEW_SAMPLE_CHECK_XLSX = os.path.join(VALIDATION_DIR, "review_sample_check.xlsx")
FINAL_VALIDATION_REPORT_PATH = os.path.join(VALIDATION_DIR, "final_validation_report.csv")

BERITA_MODELING_PATH = os.path.join(MODELING_DIR, "berita_modeling.csv")
ULASAN_MODELING_PATH = os.path.join(MODELING_DIR, "ulasan_modeling.csv")

print("CLEANED_DIR   :", CLEANED_DIR)
print("VALIDATION_DIR:", VALIDATION_DIR)
print("MODELING_DIR  :", MODELING_DIR)


In [ ]:
df_news = pd.read_csv(NEWS_CLEANED_PATH, encoding="utf-8-sig")
df_review = pd.read_csv(REVIEW_CLEANED_PATH, encoding="utf-8-sig")

print("Dataset berita cleaned :", df_news.shape)
print("Dataset ulasan cleaned :", df_review.shape)

print("\nDistribusi flag_relevansi_kesehatan (berita):")
display(df_news["flag_relevansi_kesehatan"].value_counts())

print("\nDistribusi place_category (ulasan):")
display(df_review["place_category"].value_counts())

print("\nDistribusi is_health_related (ulasan):")
display(df_review["is_health_related"].value_counts())


## BAGIAN 6 (dikerjakan lebih awal) — `document_id`

ID dibuat berdasarkan urutan baris dataframe **setelah proses cleaning**, yaitu urutan baris
persis seperti tersimpan di `berita_bangkalan_cleaned.csv` / `ulasan_bangkalan_cleaned.csv`.
Karena urutan file CSV tetap sama setiap kali dibaca, ID ini reproducible.


In [ ]:
def buat_document_id(df: pd.DataFrame, prefix: str, lebar: int = 6) -> pd.Series:
    """
    Membuat ID unik reproducible: <prefix> + nomor urut (zero-padded).
    Nomor urut mengikuti urutan baris df SAAT INI (urutan hasil cleaning),
    bukan diacak atau diurutkan ulang.
    """
    nomor = np.arange(1, len(df) + 1)
    return [f"{prefix}{n:0{lebar}d}" for n in nomor]

df_news = df_news.copy()
df_news["document_id"] = buat_document_id(df_news, "NEWS")

df_review = df_review.copy()
df_review["document_id"] = buat_document_id(df_review, "REV")

print("Contoh document_id berita :", df_news['document_id'].iloc[0], "...", df_news['document_id'].iloc[-1])
print("Contoh document_id ulasan :", df_review['document_id'].iloc[0], "...", df_review['document_id'].iloc[-1])
assert df_news["document_id"].is_unique
assert df_review["document_id"].is_unique


## BAGIAN 5 (dikerjakan lebih awal) — Kolom Waktu

Tanggal asli **tidak dihapus**. Baris dengan tanggal tidak valid tetap dipertahankan dan diberi flag.


In [ ]:
def tambah_kolom_waktu(df: pd.DataFrame, kolom_tanggal: str, flag_valid_col: str = None) -> pd.DataFrame:
    """
    Menambahkan kolom tahun, bulan, tahun_bulan (format YYYY-MM) dari kolom tanggal.
    Tidak menghapus/mengubah kolom tanggal asli. Jika flag_valid_col belum ada,
    kolom itu dibuat di sini berdasarkan hasil parsing.
    """
    df = df.copy()
    parsed = pd.to_datetime(df[kolom_tanggal], errors="coerce", format="mixed")

    nama_flag = flag_valid_col or "tanggal_valid"
    if nama_flag not in df.columns:
        df[nama_flag] = parsed.notna()

    df["tahun"] = parsed.dt.year
    df["bulan"] = parsed.dt.month
    df["tahun_bulan"] = parsed.dt.strftime("%Y-%m")  # NaT -> NaN otomatis, baris tetap ada

    jumlah_invalid = parsed.isna().sum()
    print(f"tambah_kolom_waktu('{kolom_tanggal}'): {jumlah_invalid} baris tanggal tidak valid "
          f"(baris TETAP dipertahankan, tahun/bulan/tahun_bulan = NaN untuk baris ini).")
    return df

# Berita: kolom 'tanggal' & flag 'tanggal_valid' sudah ada dari notebook 01
df_news = tambah_kolom_waktu(df_news, kolom_tanggal="tanggal", flag_valid_col="tanggal_valid")

# Ulasan: kolom 'review_date', flag tanggal_valid belum ada -> dibuat di sini
df_review = tambah_kolom_waktu(df_review, kolom_tanggal="review_date", flag_valid_col="tanggal_valid")

display(df_news[["document_id", "tanggal", "tahun", "bulan", "tahun_bulan", "tanggal_valid"]].head(3))
display(df_review[["document_id", "review_date", "tahun", "bulan", "tahun_bulan", "tanggal_valid"]].head(3))


## BAGIAN 1 — Validasi Berita Secara Manual (`perlu_pemeriksaan_manual`)

Diambil semua baris berita dengan `flag_relevansi_kesehatan == "perlu_pemeriksaan_manual"`.
Kolom `label_relevansi_final` dan `alasan_label` **sengaja dikosongkan** — tidak ada keputusan
otomatis di sini.


In [ ]:
# PENTING: jika file ini sudah ada (artinya sudah/sedang diisi manual), JANGAN ditimpa.
# Ini membuat notebook aman dijalankan ulang (mis. "Run all" di Colab) tanpa menghapus
# pekerjaan validasi manual yang sudah dilakukan. Hapus filenya sendiri secara sengaja
# jika benar-benar ingin regenerasi ulang dari nol.
if os.path.exists(BERITA_MANUAL_XLSX):
    print(f"File sudah ada, TIDAK ditimpa: {BERITA_MANUAL_XLSX}")
    print("(Hapus file ini secara manual jika ingin membuat ulang dari nol.)")
    df_manual_review = pd.read_excel(BERITA_MANUAL_XLSX)
else:
    df_manual_review = df_news.loc[
        df_news["flag_relevansi_kesehatan"] == "perlu_pemeriksaan_manual",
        ["document_id", "judul_asli", "teks_bersih", "kata_kunci", "publisher", "tanggal", "kecamatan", "url"],
    ].rename(columns={"document_id": "annotation_id"}).reset_index(drop=True)

    # Kolom yang harus diisi manual -- dikosongkan dengan tipe string agar tidak
    # berubah jadi NaN float saat dibuka/disimpan ulang di Excel.
    df_manual_review["label_relevansi_final"] = ""
    df_manual_review["alasan_label"] = ""

    df_manual_review.to_excel(BERITA_MANUAL_XLSX, index=False)
    df_manual_review.to_csv(BERITA_MANUAL_CSV, index=False, encoding="utf-8-sig")

    print(f"Tersimpan {len(df_manual_review)} baris ke:")
    print(" -", BERITA_MANUAL_XLSX)
    print(" -", BERITA_MANUAL_CSV)

display(df_manual_review.head(3))


**Cara mengisi `berita_manual_review.xlsx`:**

1. Buka file di Excel/Google Sheets. Baca `judul_asli` dan `teks_bersih` untuk tiap baris.
2. Isi kolom **`label_relevansi_final`** dengan salah satu dari dua nilai berikut PERSIS
   (huruf kecil semua, pakai garis bawah): `relevan` atau `tidak_relevan`.
   - `relevan`: isi berita membahas kondisi/isu yang berkaitan dengan kesehatan masyarakat
     atau kesehatan lingkungan (termasuk sampah/limbah/pencemaran/sanitasi *jika* dampaknya
     terhadap kesehatan masyarakat disebut/tersirat secara wajar).
   - `tidak_relevan`: isu lingkungan/umum yang tidak berkaitan dengan kesehatan masyarakat
     (mis. murni tentang mangrove sebagai isu konservasi tanpa singgungan kesehatan).
3. Isi kolom **`alasan_label`** dengan 1 kalimat singkat alasan Anda memilih label tersebut
   (akan dipakai sebagai jejak audit/metodologi, bukan wajib panjang).
4. Jangan mengubah kolom lain atau urutan baris (kolom `annotation_id` dipakai untuk
   menggabungkan kembali hasil Anda ke dataset utama pada Bagian 7).
5. Simpan file (tetap dalam format `.xlsx`, tetap di folder `data/validation/`).


## BAGIAN 2 — Pemeriksaan Sampel Heuristik (Quality Control)

Sampel ini HANYA untuk memeriksa apakah heuristik `kemungkinan_relevan` /
`kemungkinan_tidak_relevan` masuk akal. **Tidak mengubah label final secara otomatis** —
jika Anda menemukan heuristik keliru pada sampel ini, catat `document_id`-nya dan masukkan
ke `override_sample_check` pada Bagian 7.


In [ ]:
def ambil_sampel(df: pd.DataFrame, n: int, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    """Sampel acak sederhana, otomatis dibatasi jika n lebih besar dari data yang tersedia."""
    n_aman = min(n, len(df))
    if n_aman < n:
        print(f"  (diminta {n}, tersedia hanya {len(df)} baris -> diambil semua {n_aman})")
    return df.sample(n=n_aman, random_state=random_state)

sampel_relevan = ambil_sampel(
    df_news[df_news["flag_relevansi_kesehatan"] == "kemungkinan_relevan"], n=50
)
sampel_tidak_relevan = ambil_sampel(
    df_news[df_news["flag_relevansi_kesehatan"] == "kemungkinan_tidak_relevan"], n=30
)

df_sample_check = pd.concat([sampel_relevan, sampel_tidak_relevan], ignore_index=False)
df_sample_check = df_sample_check[[
    "document_id", "judul_asli", "teks_bersih", "flag_relevansi_kesehatan",
    "kata_kunci", "publisher", "tanggal", "kecamatan", "url",
]].rename(columns={"document_id": "annotation_id"})
df_sample_check["setuju_dengan_heuristik"] = ""   # diisi manual: ya / tidak
df_sample_check["catatan_qc"] = ""                 # opsional

df_sample_check.to_excel(BERITA_SAMPLE_CHECK_XLSX, index=False)
print(f"Tersimpan {len(df_sample_check)} baris "
      f"({len(sampel_relevan)} kemungkinan_relevan + {len(sampel_tidak_relevan)} kemungkinan_tidak_relevan) ke:")
print(" -", BERITA_SAMPLE_CHECK_XLSX)


## BAGIAN 3 — Validasi Label Review (Sampel)

Sampel terstratifikasi berdasarkan `place_category` di dalam masing-masing kelompok
`is_health_related`, agar sampel tidak didominasi satu kategori tempat saja.


In [ ]:
def sampel_terstratifikasi(df: pd.DataFrame, n: int, kolom_strata: str,
                            random_state: int = RANDOM_STATE) -> pd.DataFrame:
    """
    Mengambil sampel sebanyak n dari df, dialokasikan proporsional menurut
    frekuensi nilai kolom_strata, supaya sampel mencakup kategori yang beragam
    (bukan didominasi kategori terbanyak saja).
    """
    if df.empty or n <= 0:
        return df.iloc[0:0]

    proporsi = df[kolom_strata].value_counts(normalize=True)
    alokasi = (proporsi * n).round().astype(int)

    bagian = []
    for nilai_strata, jumlah in alokasi.items():
        subset = df[df[kolom_strata] == nilai_strata]
        jumlah_aman = min(jumlah, len(subset))
        if jumlah_aman > 0:
            bagian.append(subset.sample(n=jumlah_aman, random_state=random_state))

    hasil = pd.concat(bagian) if bagian else df.iloc[0:0]

    # Jika akibat pembulatan hasil masih kurang dari n, tambahkan sisa secara acak
    if len(hasil) < n:
        sisa = df.drop(hasil.index)
        tambahan_n = min(n - len(hasil), len(sisa))
        if tambahan_n > 0:
            hasil = pd.concat([hasil, sisa.sample(n=tambahan_n, random_state=random_state)])

    return hasil.sample(frac=1, random_state=random_state)  # acak urutan tampilan


mask_true = df_review["is_health_related"].astype(str).str.lower() == "true"
sampel_review_true = sampel_terstratifikasi(df_review[mask_true], n=50, kolom_strata="place_category")
sampel_review_false = sampel_terstratifikasi(df_review[~mask_true], n=30, kolom_strata="place_category")

print("Distribusi place_category pada sampel is_health_related=True:")
display(sampel_review_true["place_category"].value_counts())
print("\nDistribusi place_category pada sampel is_health_related=False:")
display(sampel_review_false["place_category"].value_counts())


In [ ]:
df_review_sample_check = pd.concat([sampel_review_true, sampel_review_false], ignore_index=False)
df_review_sample_check = df_review_sample_check[[
    "document_id", "place_name", "place_category", "review_text_bersih",
    "rating", "review_date", "is_health_related", "matched_keywords",
]].rename(columns={"document_id": "annotation_id"})
df_review_sample_check["validasi_manual"] = ""  # diisi manual: relevan / tidak_relevan

df_review_sample_check.to_excel(REVIEW_SAMPLE_CHECK_XLSX, index=False)
print(f"Tersimpan {len(df_review_sample_check)} baris "
      f"({len(sampel_review_true)} True + {len(sampel_review_false)} False) ke:")
print(" -", REVIEW_SAMPLE_CHECK_XLSX)
display(df_review_sample_check.head(3))


## BAGIAN 4 — Analisis Panjang Review & Ambang Batas Modeling

Dihitung berdasarkan jumlah karakter dan jumlah kata pada `review_text_bersih`.
**Tidak ada baris yang dihapus** — hanya kolom flag `teks_layak_modeling` yang ditambahkan.


In [ ]:
df_review["panjang_karakter"] = df_review["review_text_bersih"].fillna("").astype(str).str.len()
df_review["jumlah_kata"] = df_review["review_text_bersih"].fillna("").astype(str).str.split().str.len()

def bucket_kata(n_kata: int) -> str:
    if n_kata <= 2:
        return "0-2 kata"
    elif n_kata <= 4:
        return "3-4 kata"
    elif n_kata <= 9:
        return "5-9 kata"
    else:
        return ">=10 kata"

df_review["bucket_panjang"] = df_review["jumlah_kata"].apply(bucket_kata)

print("=== Statistik panjang karakter ===")
display(df_review["panjang_karakter"].describe())

print("\n=== Statistik jumlah kata ===")
display(df_review["jumlah_kata"].describe())

print("\n=== Distribusi bucket panjang (jumlah kata) ===")
urutan_bucket = ["0-2 kata", "3-4 kata", "5-9 kata", ">=10 kata"]
display(df_review["bucket_panjang"].value_counts().reindex(urutan_bucket, fill_value=0))


In [ ]:
# --- Rekomendasi ambang batas ---
# ASUMSI METODOLOGIS: review dengan 0-2 kata (mis. "bagus", "oke", "mantap") umumnya
# tidak membawa informasi topik yang bisa dibedakan BERTopic (representasi SBERT-nya
# nyaris tidak informatif untuk clustering topik). Review >=3 kata biasanya sudah
# menyebut objek/keluhan spesifik ("wc kotor", "airnya keruh dan bau").
# Karena itu ambang MIN_KATA_REVIEW = 3 diusulkan sebagai default, BUKAN keputusan mutlak --
# angka ini bisa diubah dan didokumentasikan ulang jika ditemukan alasan lain.
MIN_KATA_REVIEW = 3

df_review["teks_layak_modeling"] = (
    (~df_review["teks_kosong"]) & (df_review["jumlah_kata"] >= MIN_KATA_REVIEW)
)

print(f"Ambang batas yang dipakai: MIN_KATA_REVIEW = {MIN_KATA_REVIEW} kata")
print(f"Review yang lolos flag teks_layak_modeling : {df_review['teks_layak_modeling'].sum()} dari {len(df_review)}")
print(f"Review yang TIDAK lolos                    : {(~df_review['teks_layak_modeling']).sum()}")
print("\n(Dataset cleaned TIDAK diubah/dipangkas -- hanya kolom flag baru yang ditambahkan.)")


## BAGIAN 7 — Menggabungkan Hasil Validasi Manual & Menentukan Label Final Berita

Jalankan sel ini **setelah** Anda mengisi `berita_manual_review.xlsx` (Bagian 1) dan,
jika perlu, mencatat hasil pemeriksaan sampel (Bagian 2) ke `override_sample_check` di bawah.

**Mekanisme keputusan (eksplisit & terdokumentasi):**
1. Default `kemungkinan_relevan` -> `relevan` (asumsi: heuristik kata kunci kuat cukup andal).
2. Default `kemungkinan_tidak_relevan` -> `tidak_relevan` (asumsi: tidak ada kata kunci kesehatan
   sama sekali).
3. `perlu_pemeriksaan_manual` -> HANYA diisi dari hasil `berita_manual_review.xlsx`; jika belum
   diisi, baris tersebut TETAP tanpa label final dan TIDAK masuk ke `berita_modeling.csv`.
4. `override_sample_check` (opsional) -> menimpa langkah 1/2 di atas untuk `document_id`
   tertentu yang terbukti keliru saat pemeriksaan sampel (Bagian 2).

Setiap baris menyimpan jejak keputusan pada kolom `sumber_keputusan`.


In [ ]:
def muat_validasi_manual(path_xlsx: str) -> pd.DataFrame:
    """Membaca file validasi manual dan memvalidasi isian yang sudah diberi label."""
    df_manual = pd.read_excel(path_xlsx)
    df_manual["label_relevansi_final"] = df_manual["label_relevansi_final"].astype(str).str.strip()

    nilai_valid = {"relevan", "tidak_relevan"}
    sudah_diisi = df_manual[df_manual["label_relevansi_final"].isin(nilai_valid)]
    belum_diisi = df_manual[~df_manual["label_relevansi_final"].isin(nilai_valid)]

    print(f"muat_validasi_manual: {len(sudah_diisi)} dari {len(df_manual)} baris sudah diisi label valid.")
    if len(belum_diisi) > 0:
        print(f"  {len(belum_diisi)} baris masih kosong/belum divalidasi -- baris ini akan "
              f"TETAP tanpa label final dan tidak masuk ke corpus modeling.")
    return df_manual


def terapkan_keputusan_relevansi_berita(df_news: pd.DataFrame, df_manual: pd.DataFrame,
                                         override_sample_check: dict = None) -> pd.DataFrame:
    """
    Menentukan label_relevansi_final untuk SELURUH dataset berita berdasarkan
    mekanisme yang didokumentasikan di atas sel ini.
    """
    df = df_news.copy()
    df["label_relevansi_final"] = pd.NA
    df["sumber_keputusan"] = ""

    # 1) Default heuristik kuat
    m_relevan = df["flag_relevansi_kesehatan"] == "kemungkinan_relevan"
    df.loc[m_relevan, "label_relevansi_final"] = "relevan"
    df.loc[m_relevan, "sumber_keputusan"] = "default_heuristik_kata_kunci_kuat"

    m_tidak_relevan = df["flag_relevansi_kesehatan"] == "kemungkinan_tidak_relevan"
    df.loc[m_tidak_relevan, "label_relevansi_final"] = "tidak_relevan"
    df.loc[m_tidak_relevan, "sumber_keputusan"] = "default_heuristik_tanpa_kata_kunci"

    # 2) Hasil validasi manual (paling otoritatif untuk kasus ambigu)
    nilai_valid = {"relevan", "tidak_relevan"}
    manual_terisi = df_manual[df_manual["label_relevansi_final"].isin(nilai_valid)]
    manual_map = manual_terisi.set_index("annotation_id")["label_relevansi_final"]

    idx_manual = df["document_id"].isin(manual_map.index)
    df.loc[idx_manual, "label_relevansi_final"] = df.loc[idx_manual, "document_id"].map(manual_map)
    df.loc[idx_manual, "sumber_keputusan"] = "manual_review_perlu_pemeriksaan"

    # Baris perlu_pemeriksaan_manual yang BELUM diisi -> catat sebagai belum tervalidasi
    m_belum = (df["flag_relevansi_kesehatan"] == "perlu_pemeriksaan_manual") & (~idx_manual)
    df.loc[m_belum, "sumber_keputusan"] = "belum_divalidasi_manual"

    # 3) Override eksplisit dari sample QC (Bagian 2), jika ada
    if override_sample_check:
        for doc_id, label in override_sample_check.items():
            assert label in nilai_valid, f"Label tidak valid untuk {doc_id}: {label}"
            df.loc[df["document_id"] == doc_id, "label_relevansi_final"] = label
            df.loc[df["document_id"] == doc_id, "sumber_keputusan"] = "override_manual_dari_sample_qc"

    return df


# --- Contoh pemakaian ---
# Isi dict ini HANYA jika Anda menemukan heuristik keliru saat memeriksa
# berita_sample_check.xlsx (Bagian 2). Kosongkan {} jika tidak ada temuan.
override_sample_check_berita = {
    # "NEWS000123": "tidak_relevan",
}

if os.path.exists(BERITA_MANUAL_XLSX):
    df_manual_terisi = muat_validasi_manual(BERITA_MANUAL_XLSX)
    df_news = terapkan_keputusan_relevansi_berita(df_news, df_manual_terisi, override_sample_check_berita)
    print("\nDistribusi label_relevansi_final (berita):")
    display(df_news["label_relevansi_final"].value_counts(dropna=False))
    print("\nDistribusi sumber_keputusan (berita):")
    display(df_news["sumber_keputusan"].value_counts())
else:
    print(f"File {BERITA_MANUAL_XLSX} belum ditemukan. Isi dulu Bagian 1 sebelum melanjutkan.")


## BAGIAN 8 — Corpus Review (Keputusan Relevansi Ulasan)

Default label ulasan memakai `is_health_related` (dibuat berdasarkan daftar kata kunci saat
scraping) sebagai titik awal, ditimpa oleh temuan pada `review_sample_check.xlsx` bila ada.
`place_category == "Tempat Umum"` TIDAK otomatis membuat ulasan tidak relevan -- keputusan
tetap mengikuti `is_health_related` + override manual, bukan `place_category`.


In [ ]:
def terapkan_keputusan_relevansi_review(df_review: pd.DataFrame,
                                         override_sample_check: dict = None) -> pd.DataFrame:
    df = df_review.copy()
    is_true = df["is_health_related"].astype(str).str.lower() == "true"
    df["label_relevansi_final"] = np.where(is_true, "relevan", "tidak_relevan")
    df["sumber_keputusan"] = "default_is_health_related_keyword"

    if override_sample_check:
        for doc_id, label in override_sample_check.items():
            assert label in ("relevan", "tidak_relevan"), f"Label tidak valid untuk {doc_id}: {label}"
            df.loc[df["document_id"] == doc_id, "label_relevansi_final"] = label
            df.loc[df["document_id"] == doc_id, "sumber_keputusan"] = "override_manual_dari_sample_qc"

    return df


# Isi dict ini HANYA jika ditemukan kekeliruan saat memeriksa review_sample_check.xlsx
override_sample_check_review = {
    # "REV000456": "relevan",
}

df_review = terapkan_keputusan_relevansi_review(df_review, override_sample_check_review)

print("Distribusi label_relevansi_final (ulasan):")
display(df_review["label_relevansi_final"].value_counts())

# Kriteria masuk corpus modeling: relevan + teks tidak kosong + lolos ambang panjang (Bagian 4)
mask_review_modeling = (
    (df_review["label_relevansi_final"] == "relevan")
    & (~df_review["teks_kosong"])
    & (df_review["teks_layak_modeling"])
)
print(f"\nUlasan yang memenuhi SEMUA kriteria corpus modeling: {mask_review_modeling.sum()} dari {len(df_review)}")


## BAGIAN 9 & 10 — Format Corpus Final (Dua Corpus Terpisah)

`teks_bersih` (berita) dan `review_text_bersih` (ulasan) menjadi kolom `text`.
`reviewer_name`, alamat lengkap, dan metadata lain **tidak** dimasukkan ke kolom `text` --
metadata tetap tersimpan sebagai kolom terpisah untuk dipakai setelah topic modeling.
Berita dan ulasan **tidak digabung** -- keduanya tetap dua file terpisah agar bisa dimodelkan
secara independen dengan pipeline SBERT + BERTopic yang sama.


In [ ]:
# --- Corpus berita ---
mask_berita_modeling = df_news["label_relevansi_final"] == "relevan"

berita_modeling = df_news.loc[mask_berita_modeling, [
    "document_id", "teks_bersih", "tanggal", "tahun", "bulan", "tahun_bulan",
    "publisher", "kecamatan", "url",
]].rename(columns={"teks_bersih": "text"})
berita_modeling.insert(1, "source", "berita")

berita_modeling.to_csv(BERITA_MODELING_PATH, index=False, encoding="utf-8-sig")
print(f"berita_modeling.csv: {len(berita_modeling)} baris (dari {len(df_news)} baris cleaned) -> {BERITA_MODELING_PATH}")


In [ ]:
# --- Corpus ulasan ---
ulasan_modeling = df_review.loc[mask_review_modeling, [
    "document_id", "review_text_bersih", "review_date", "tahun", "bulan", "tahun_bulan",
    "place_name", "place_category", "rating",
]].rename(columns={"review_text_bersih": "text", "review_date": "tanggal"})
ulasan_modeling.insert(1, "source", "ulasan")

ulasan_modeling.to_csv(ULASAN_MODELING_PATH, index=False, encoding="utf-8-sig")
print(f"ulasan_modeling.csv: {len(ulasan_modeling)} baris (dari {len(df_review)} baris cleaned) -> {ULASAN_MODELING_PATH}")

assert "reviewer_name" not in ulasan_modeling.columns
assert "place_address" not in ulasan_modeling.columns
print("\nKolom berita_modeling.csv:", list(berita_modeling.columns))
print("Kolom ulasan_modeling.csv:", list(ulasan_modeling.columns))


## BAGIAN 11 — Laporan Kualitas Akhir

In [ ]:
# Ambil jumlah data awal (sebelum cleaning) dari log yang disimpan notebook 01
if os.path.exists(CLEANING_LOG_PATH):
    with open(CLEANING_LOG_PATH, encoding="utf-8") as f:
        cleaning_log = json_lib.load(f)
    awal_berita = cleaning_log["dataset_berita"]["baris_sebelum"]
    cleaned_berita = cleaning_log["dataset_berita"]["baris_sesudah"]
    awal_review = cleaning_log["dataset_ulasan"]["baris_sebelum"]
    cleaned_review = cleaning_log["dataset_ulasan"]["baris_sesudah"]
else:
    print(f"PERINGATAN: {CLEANING_LOG_PATH} tidak ditemukan, memakai jumlah baris "
          f"dataset cleaned sebagai jumlah awal (tidak akurat untuk 'jumlah_data_awal').")
    awal_berita = cleaned_berita = len(df_news)
    awal_review = cleaned_review = len(df_review)

modeling_berita = len(berita_modeling)
modeling_review = len(ulasan_modeling)

laporan = pd.DataFrame([
    {
        "dataset": "berita",
        "jumlah_data_awal": awal_berita,
        "jumlah_data_cleaned": cleaned_berita,
        "jumlah_data_modeling": modeling_berita,
        "jumlah_dibuang": cleaned_berita - modeling_berita,
        "alasan_pembuangan": (
            "label_relevansi_final != 'relevan' (termasuk baris perlu_pemeriksaan_manual "
            "yang belum divalidasi di berita_manual_review.xlsx)"
        ),
    },
    {
        "dataset": "ulasan",
        "jumlah_data_awal": awal_review,
        "jumlah_data_cleaned": cleaned_review,
        "jumlah_data_modeling": modeling_review,
        "jumlah_dibuang": cleaned_review - modeling_review,
        "alasan_pembuangan": (
            f"is_health_related=False (tanpa override manual) DAN/ATAU teks_kosong=True "
            f"DAN/ATAU jumlah_kata < {MIN_KATA_REVIEW}"
        ),
    },
])

laporan.to_csv(FINAL_VALIDATION_REPORT_PATH, index=False, encoding="utf-8-sig")

print("=== RINGKASAN ALUR DATA ===")
print(f"BERITA : awal {awal_berita} -> cleaned {cleaned_berita} -> modeling {modeling_berita}")
print(f"REVIEW : awal {awal_review} -> cleaned {cleaned_review} -> modeling {modeling_review}")
print()
display(laporan)
print("\nTersimpan ke:", FINAL_VALIDATION_REPORT_PATH)


## Ringkasan, Status, dan Langkah Selanjutnya

**1. Apa yang masih harus dilakukan secara manual:**
- Isi `data/validation/berita_manual_review.xlsx` (kolom `label_relevansi_final` &
  `alasan_label`) untuk seluruh baris `perlu_pemeriksaan_manual`.
- Periksa `data/validation/berita_sample_check.xlsx` dan `data/validation/review_sample_check.xlsx`
  sebagai quality control; jika heuristik keliru, catat `document_id`-nya ke
  `override_sample_check_berita` / `override_sample_check_review` pada Bagian 7 & 8, lalu
  jalankan ulang notebook dari Bagian 7 ke bawah.
- Setelah itu jalankan ulang seluruh notebook (atau minimal dari Bagian 7) agar
  `berita_modeling.csv`, `ulasan_modeling.csv`, dan `final_validation_report.csv` terbentuk
  dari label final yang sudah divalidasi, bukan hanya default heuristik.

**2. Kapan dataset dianggap siap untuk SBERT-BERTopic:**
- Semua baris `perlu_pemeriksaan_manual` sudah punya `label_relevansi_final` terisi (tidak ada
  lagi `sumber_keputusan == "belum_divalidasi_manual"` pada `df_news`).
- Sampel QC (Bagian 2 & 3) sudah diperiksa, dan temuan (jika ada) sudah dimasukkan ke
  `override_sample_check_*`.
- `final_validation_report.csv` menunjukkan angka yang masuk akal (`jumlah_dibuang` bisa
  dijelaskan sepenuhnya oleh kolom `alasan_pembuangan`).
- `berita_modeling.csv` dan `ulasan_modeling.csv` sudah dicek sekilas (`.sample()` beberapa
  baris) untuk memastikan kolom `text` tidak kosong dan tidak mengandung sisa boilerplate.

**3. Struktur input untuk `02_sbert_bertopic.ipynb`:**
- Baca `data/modeling/berita_modeling.csv` dan `data/modeling/ulasan_modeling.csv` secara terpisah.
- Kolom yang dipakai sebagai input embedding SBERT: **`text`** (bukan kolom lain).
- Kolom lain (`document_id`, `source`, `tanggal`, `tahun`, `bulan`, `tahun_bulan`, `publisher`/
  `place_name`, `kecamatan`/`place_category`, `rating`, `url`) disimpan sebagai metadata untuk
  dipakai SETELAH topic modeling (mis. tren topik per `tahun_bulan`, per `kecamatan`, atau
  perbandingan topik berita vs ulasan).
- Kedua corpus dimodelkan terpisah dengan pipeline SBERT + BERTopic yang sama; hasilnya baru
  dibandingkan pada tahap dashboard, bukan digabung sebagai satu corpus tunggal.
